# SOP-03: The Scrambled Null Trap and Alanine Scan: Why High Computer Scores Can Lie
## Streptococcus mutans GTF-SI Macrocyclic Inhibitor Campaign
**Standard Operating Procedure ID:** SOP-03  
**Principal Investigator:** Dr. Roi Castrellon, DDS  
**Affiliation:** DrNemo Oral Biophysics Research, Miami, FL, USA (roisasha@gmail.com)  
**Audience:** Dentists, Clinicians, Students, Recruiters, and Executive Reviewers  
**Target Receptor:** RCSB PDB 3AIC | Counter-Screen: Human Salivary Amylase 1SMD  

---

### Plain-English "Why This Matters"
A dangerous pitfall in computational drug discovery is the **false positive trap**. If a researcher takes a random string of greasy amino acids and docks it into a protein pocket, the computer will often return an attractive score like `-8.0 kcal/mol` simply because large aromatic molecules can pack into hydrophobic crevices.

To prevent fooling ourselves, I executed two statistical and genetic sanity checks:
1. **The 39 Scrambled Sequence Null Test:** Taking the exact same amino acids from the candidate, shuffling them like an anagram into 39 random cyclic permutations, and docking every single one. If scrambled sequences score just as well, static docking alone cannot prove sequence specificity.
2. **The 7-Site Alanine Scan:** Replacing each amino acid one by one with Alanine (`A`). In principle this reveals which residues serve as catalytic anchors and exposes traps where higher docking scores actually destroy saliva survival.

## 1. The Scrambled Null Test: Failing to Reject H0
To construct the statistical null distribution, the two outer Cysteine residues (`Cys1` and `Cys9`) were preserved to maintain the macrocyclic ring, while the seven interior residues were permuted into $N=39$ scrambled cyclic nonapeptides.

### The Empirical Null Distribution
- **Null Mean ($\mu$):** $-8.343 \pm 0.470\text{ kcal/mol}$
- **Lead Candidate (`CAHWRPWHC`):** $-8.191\text{ kcal/mol}$
- **Z-Score:** $+0.32\sigma$
- **p-Value:** $p = 0.5897$

### Scientific Conclusion
Because $p = 0.59 \gg 0.05$, the lead candidate **fails to reject the null hypothesis ($H_0$)**! 
Random scrambled permutations of the exact same composition bind with comparable static docking energy. This proves that static rigid-receptor docking cannot establish sequence-specific binding in the wide 28 A cleft of GTF-SI. In accordance with Karl Popper's demarcation principle, this angle is classified as **`INCONCLUSIVE_WITHIN_NOISE`**.

## 2. In-Silico Alanine Hotspot Scanning
In an alanine scan, every non-alanine position is systematically mutated to alanine (`A`). 
Alanine has a minimal methyl side chain (`-CH3`), effectively removing side-chain functional groups:

The scan is parented on the **wild type** `CPHWRPWHC` ($-8.062\text{ kcal/mol}$), of which the lead `CAHWRPWHC` is itself the `P2A` mutant. Deltas below are against that wild type. A positive delta is a loss of affinity; the hotspot threshold set in advance is $+0.70\text{ kcal/mol}$.

| Mutant | Sequence | Vina Score | $\Delta\Delta G$ vs wild type | Hotspot? | Role in Binding |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **P2A (the lead)** | `CAHWRPWHC` | $-8.191\text{ kcal/mol}$ | $-0.129$ | No | Conformational hinge; gains affinity |
| **H3A** | `CPAWRPWHC` | $-7.869\text{ kcal/mol}$ | $+0.193$ | No | Acid-switch histidine |
| **W4A** | `CPHARPWHC` | $-8.269\text{ kcal/mol}$ | $-0.207$ | No | Pi-stacking platform; gains affinity |
| **R5A** | `CPHWAPWHC` | $-8.531\text{ kcal/mol}$ | $-0.469$ | No | Salt-bridge partner; gains affinity |
| **P6A** | `CPHWRAWHC` | $-8.890\text{ kcal/mol}$ | $-0.828$ | No | **Affinity trap: opens a tryptic site at Arg5** |
| **W7A** | `CPHWRPAHC` | $-7.805\text{ kcal/mol}$ | $+0.257$ | No | Lipophilic cleft wall; largest loss in the scan |
| **H8A** | `CPHWRPWAC` | $-7.978\text{ kcal/mol}$ | $+0.084$ | No | Second acid-switch histidine |

**No position is a hotspot.** The largest affinity loss anywhere in the scan is `W7A` at $+0.257\text{ kcal/mol}$, about a third of the threshold, so the scan identifies no energetic anchor. Three substitutions (`W4A`, `R5A`, `P6A`) *improve* the docking score, which is itself evidence that the score is not reading sequence-specific contacts: deleting a tryptophan, an arginine or a proline from a nonapeptide should not improve a specific interaction.

### The P6A Affinity Trap
Mutant `P6A` achieves a stronger docking score ($-8.890\text{ kcal/mol}$). 
However, mutating Proline-6 to Alanine destroys the `Arg5-Pro6` steric shield. In saliva, trypsin specifically cleaves after Arginine (`R`) unless it is immediately followed by Proline (`P`). `P6A` would be destroyed by salivary enzymes within minutes.

In [ ]:
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from pathlib import Path

# Publication aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 120

repo_root = Path("..").resolve() if Path("../data").exists() else Path(".").resolve()
benchmark_path = repo_root / "demo_data/results/falsification_benchmark.json"

with open(benchmark_path, "r", encoding="utf-8") as f:
    bm = json.load(f)

null_test = bm["scrambled_null_test"]
null_scores = [p["vina_3aic_kcal"] for p in null_test["permutations"]]
null_mean = null_test["mean_kcal"]
null_std = null_test["std_kcal"]
lead_score = -8.191
z_score = (lead_score - null_mean) / null_std
p_val = 1 - norm.cdf(abs(z_score))

print(f"Scrambled Sequence Null Statistics (N={len(null_scores)}):")
print(f"  Null Distribution Mean: {null_mean:.3f} +/- {null_std:.3f} kcal/mol")
print(f"  Lead Score (CAHWRPWHC): {lead_score:.3f} kcal/mol")
print(f"  Z-score: {z_score:+.2f} sigma")
print(f"  Two-tailed p-value: {p_val:.4f} (Fails to reject H0: p > 0.05)")

# Plot Scrambled Null Histogram
fig, ax = plt.subplots(figsize=(8, 4.8))
n_bins, bins, patches = ax.hist(null_scores, bins=12, density=True, color="#94a3b8", edgecolor="black", alpha=0.75, label=f"Scrambled Isomers (N={len(null_scores)})")

# Overlay Gaussian fit
x = np.linspace(min(null_scores)-0.5, max(null_scores)+0.5, 200)
ax.plot(x, norm.pdf(x, null_mean, null_std), color="#3b82f6", linewidth=2, label=f"Null Gaussian Model (mean={null_mean:.2f})")

# Vertical indicators
ax.axvline(lead_score, color="#10b981", linewidth=2.5, linestyle="-", label=f"Lead CAHWRPWHC (-8.19 kcal/mol, Z={z_score:+.2f})")
ax.axvline(null_mean, color="#3b82f6", linewidth=1.5, linestyle="--", label="Null Mean (-8.34 kcal/mol)")

ax.set_xlabel("Vina GTF-SI Binding Free Energy (3AIC, kcal/mol)")
ax.set_ylabel("Probability Density")
ax.set_title("Karl Popper Statistical Null Distribution: H0 Not Rejected (p = 0.5897)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Alanine Scan Energetic Shifts
ala_data = bm["alanine_scan"]
ala_meta = bm["alanine_scan_meta"]
# Parent the scan where the data is parented: on the wild type, not on the lead.
parent_score = ala_meta["parent_affinity_kcal"]
mutants = list(ala_data.keys())
scores = [ala_data[m]["vina_3aic_kcal"] for m in mutants]
deltas = [ala_data[m]["delta_binding_affinity_kcal"] for m in mutants]
assert all(abs((s - parent_score) - d) < 1e-6 for s, d in zip(scores, deltas)), \
    "stored deltas do not match the stored parent affinity"
print(f"Alanine scan parent: {ala_meta['parent_sequence']} ({parent_score:+.3f} kcal/mol)")
print(f"Largest affinity loss: {ala_meta['largest_affinity_loss_kcal']:+.3f} kcal/mol "
      f"against a {ala_meta['hotspot_threshold_kcal']:+.2f} threshold; "
      f"hotspots identified: {ala_meta['identified_hotspots'] or 'none'}")

fig, ax = plt.subplots(figsize=(8, 4.5))
colors = ["#10b981" if d > 0.15 else ("#ef4444" if d < -0.5 else "#3b82f6") for d in deltas]

bars = ax.bar(mutants, deltas, color=colors, edgecolor="black", width=0.5, alpha=0.85)
ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Delta-Delta G vs Lead (kcal/mol)")
ax.set_title("Alanine Scan vs Wild Type: No Hotspot Reaches the +0.70 Threshold")

for bar in bars:
    y = bar.get_height()
    offset = 0.05 if y >= 0 else -0.08
    ax.text(bar.get_x() + bar.get_width()/2, y + offset, f"{y:+.2f}", ha="center", fontsize=9, fontweight="bold")

# Mark the hotspot threshold the scan is judged against, and the one real liability.
ax.axhline(0.70, color="#059669", linestyle="--", linewidth=1.5, label="Hotspot threshold +0.70 kcal/mol")
ax.text(len(mutants) - 1, 0.72, "no bar reaches this line: no hotspot", ha="right",
        color="#059669", fontweight="bold", fontsize=9)
ax.text(4, -0.85, "P6A trap\n(opens a trypsin site)", ha="center", color="#dc2626", fontweight="bold", fontsize=9)
ax.legend(loc="upper left", fontsize=8)

ax.set_ylim(-1.0, 0.95)
plt.tight_layout()
plt.show()

### Q&A
**Q1: Why is failing to reject H0 in the scrambled test a good thing to report?**  
*Answer:* It upholds scientific integrity. It prevents publishing false claims that a specific peptide sequence is unique when random scrambles score equally well in static docking. It points directly to the need for dynamic wet-lab kinetic confirmation.

**Q2: Why not select P6A if it docks with a stronger score of -8.89 kcal/mol?**  
*Answer:* Selecting P6A is the quintessential single-objective blunder. While docking score improves, Proline-6 is required to shield Arginine-5 from salivary trypsin. In saliva, P6A is cleaved and inactivated.

### Data Analysis Key Findings
- **Static Docking Limitation:** The scrambled null distribution demonstrates that rigid-receptor docking alone cannot verify sequence specificity in GTF-SI (Z = +0.32, p = 0.5897).
- **No Catalytic Anchors:** alanine scanning of the wild type `CPHWRPWHC` finds no energetic anchor. Only three of seven substitutions lose any affinity (H3A +0.193, W7A +0.257, H8A +0.084 kcal/mol), none reaching the +0.70 kcal/mol hotspot threshold. Three substitutions improve the score, which is itself a sign the score is not reading sequence-specific contacts.
- **Steric Shield Requirement:** Proline-6 is essential for trypsin resistance, proving that biological viability overrides raw docking affinity.

### Insights or Next Steps
- **Dynamic Falsification:** Molecular dynamics (MD) simulations in explicit solvent should evaluate whether CAHWRPWHC exhibits longer cleft residence times than scrambled controls.

> [!NOTE]
> **Key Takeaway Box:** High computer docking scores can mislead. The scrambled null test confirms that static docking fails to reject H0 (p=0.5897), and the alanine scan finds no energetic anchor at all, while still unmasking P6A as an affinity trap that opens a trypsin site.